# Detección de mensajes spam con red neuronal recurrente (LSTM)

**Actividad:** RNN/LSTM/GRU + Streamlit
**Dataset:** SMS Spam Collection v1 (UCI)
**Tarea:** clasificación binaria de mensajes SMS en `ham` (legítimo) o `spam`.

---

## Contenido

1. Descarga y carga del dataset
2. Análisis exploratorio
3. Limpieza y preprocesamiento
4. División en entrenamiento, validación y prueba
5. Construcción de la red LSTM
6. Entrenamiento
7. Evaluación con métricas
8. Ejemplos de predicciones
9. Conclusiones

Este notebook es autocontenido: todo lo necesario se ejecuta aquí, pero el
proyecto también incluye `train.py`, `predict.py` y `app.py` (Streamlit).

## 0. Imports y configuración

In [ ]:
import json
import random
import re
import zipfile
from collections import Counter
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

DATA_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/"
    "smsspamcollection.zip"
)
RAW_NAME = "SMSSpamCollection"
TEST_SIZE = 0.15
VAL_SIZE = 0.15
MAX_VOCAB = 10_000
MAX_LEN = 160
EMBEDDING_DIM = 128
LSTM_UNITS = 128
BATCH_SIZE = 64
EPOCHS = 20

CLASS_NAMES = ["ham", "spam"]

sns.set_theme(style="whitegrid")
print("TensorFlow:", keras.__version__)

## 1. Descarga y carga del dataset

El **SMS Spam Collection v1** es un corpus de 5.574 mensajes en inglés
recopilados por la Universidad de California (UCI) para investigar el filtrado
de spam SMS. Cada línea del fichero tiene dos columnas separadas por tabulador:
la etiqueta (`ham`/`spam`) y el texto del mensaje.

In [ ]:
import os

if not os.path.exists(RAW_NAME):
    print("Descargando el corpus...")
    with urlopen(DATA_URL, timeout=120) as respuesta:
        datos = respuesta.read()
    with open("smsspamcollection.zip", "wb") as archivo:
        archivo.write(datos)
    with zipfile.ZipFile("smsspamcollection.zip") as archivo:
        archivo.extract(RAW_NAME)
else:
    print("El corpus ya estaba descargado.")

lineas = open(RAW_NAME, encoding="utf-8").read().splitlines()
mensajes = []
etiquetas_crudas = []
for linea in lineas:
    if "\t" in linea:
        etiqueta, mensaje = linea.split("\t", 1)
        mensajes.append(mensaje)
        etiquetas_crudas.append(etiqueta.strip().lower())

# ham -> 0, spam -> 1
y = np.array([1 if e == "spam" else 0 for e in etiquetas_crudas], dtype="float32")

print(f"Mensajes: {len(mensajes)}")
print(f"ham: {(y == 0).sum()} | spam: {(y == 1).sum()}")
mensajes[:3]

## 2. Análisis exploratorio

### 2.1. Distribución de clases

El corpus está **muy desbalanceado**: solo el 13.4 % de los mensajes son spam.

In [ ]:
cuenta = np.bincount(y.astype(int), minlength=2)

fig, ax = plt.subplots(figsize=(6, 4))
barras = ax.bar(["ham", "spam"], cuenta, color=["#4c72b0", "#c44e52"])
for barra, valor in zip(barras, cuenta):
    ax.text(
        barra.get_x() + barra.get_width() / 2,
        valor + 30,
        f"{valor}\n({valor / len(y):.1%})",
        ha="center",
    )
ax.set_title("Distribución de clases")
ax.set_ylabel("Mensajes")
plt.show()

print(f"Proporción de spam: {(y == 1).mean():.1%}")

### 2.2. Longitud de los mensajes

Los mensajes de spam tienden a ser **más largos** que los legítimos, así que la
longitud es ya una señal bastante informativa antes de mirar el texto.

In [ ]:
longitud = np.array([len(m.split()) for m in mensajes])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.hist(longitud, bins=40, color="#4c72b0", edgecolor="white")
ax1.axvline(MAX_LEN, color="#c44e52", linestyle="--", label=f"MAX_LEN = {MAX_LEN}")
ax1.set_title("Longitud de los mensajes")
ax1.set_xlabel("Palabras")
ax1.legend()

ax2.boxplot(
    [longitud[y == 0], longitud[y == 1]],
    tick_labels=["ham", "spam"],
)
ax2.set_title("Longitud según clase")
plt.tight_layout()
plt.show()

print(f"Media ham:  {longitud[y == 0].mean():.1f} palabras")
print(f"Media spam: {longitud[y == 1].mean():.1f} palabras")
print(f"Percentil 95 de longitud: {np.percentile(longitud, 95):.0f} palabras")

### 2.3. Palabras más frecuentes por clase

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for clase in (0, 1):
    subset = [m for m, etiqueta in zip(mensajes, y) if etiqueta == clase]
    palabras = [p.lower() for m in subset for p in m.split()]
    top = Counter(palabras).most_common(15)
    ejes = [p for p, _ in top][::-1]
    valores = [v for _, v in top][::-1]
    axes[clase].barh(ejes, valores, color=["#4c72b0", "#c44e52"][clase])
    axes[clase].set_title(f"Palabras más frecuentes: {CLASS_NAMES[clase]}")
plt.tight_layout()
plt.show()

### 2.4. Ejemplos reales de cada clase

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 5))
for clase in (0, 1):
    indices = np.where(y == clase)[0][:2]
    for eje, indice in zip(axes[clase], indices):
        eje.axis("off")
        eje.set_title(
            f"[{CLASS_NAMES[clase]}]",
            color="#4c72b0" if clase == 0 else "#c44e52",
        )
        eje.text(0, 0.95, mensajes[indice][:150], fontsize=8, va="top", wrap=True)
plt.tight_layout()
plt.show()

## 3. Limpieza y preprocesamiento

Transformamos el texto en secuencias de enteros listas para la LSTM:

1. Todo en minúsculas.
2. URLs, correos, cantidades de dinero y números se sustituyen por marcadores
   (`<url>`, `<email>`, `<dinero>`, `<num>`) para reducir la dimensionalidad.
3. Se eliminan los signos de puntuación.
4. Un `Tokenizer` de Keras convierte cada palabra en su índice.
5. Las secuencias se rellenan con ceros hasta `MAX_LEN` (relleno por la
   izquierda, que es lo que espera una LSTM) y se recortan por la derecha.

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
DINERO_RE = re.compile(r"[$£€]\s?\d[\d.,]*")
NUMERO_RE = re.compile(r"\d+")
NO_ALFA_RE = re.compile(r"[^a-z\s]")
ESPACIOS_RE = re.compile(r"\s+")


def limpiar(texto):
    texto = texto.lower()
    texto = URL_RE.sub(" <url> ", texto)
    texto = EMAIL_RE.sub(" <email> ", texto)
    texto = DINERO_RE.sub(" <dinero> ", texto)
    texto = NUMERO_RE.sub(" <num> ", texto)
    texto = NO_ALFA_RE.sub(" ", texto)
    return ESPACIOS_RE.sub(" ", texto).strip()


mensajes_limpios = [limpiar(m) for m in mensajes]

print("Original:")
print(mensajes[0][:120])
print("\nLimpio:")
print(mensajes_limpios[0][:120])

## 4. División en entrenamiento, validación y prueba

Usamos una división **estratificada** 70 / 15 / 15 que mantiene la proporción de
spam en los tres conjuntos. El `Tokenizer` se ajusta **solo con el conjunto de
entrenamiento**, para no filtrar información del conjunto de prueba.

In [ ]:
indices = np.arange(len(mensajes_limpios))
idx_ent, idx_temp = train_test_split(
    indices, test_size=TEST_SIZE + VAL_SIZE, stratify=y, random_state=SEED
)
proporcion_val = VAL_SIZE / (VAL_SIZE + TEST_SIZE)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=1 - proporcion_val, stratify=y[idx_temp], random_state=SEED
)

tokenizer = keras.preprocessing.text.Tokenizer(
    num_words=MAX_VOCAB, filters="", oov_token="<OOV>"
)
tokenizer.fit_on_texts([mensajes_limpios[i] for i in idx_ent])


def codificar(textos):
    secuencias = tokenizer.texts_to_sequences(textos)
    return keras.preprocessing.sequence.pad_sequences(
        secuencias, maxlen=MAX_LEN, padding="pre", truncating="post"
    )


x = codificar(mensajes_limpios)
x_train, x_val, x_test = x[idx_ent], x[idx_val], x[idx_test]
y_train, y_val, y_test = y[idx_ent], y[idx_val], y[idx_test]

print(f"Entrenamiento: {x_train.shape}  spam: {int(y_train.sum())}")
print(f"Validación:    {x_val.shape}  spam: {int(y_val.sum())}")
print(f"Prueba:        {x_test.shape}  spam: {int(y_test.sum())}")
print(f"Vocabulario:   {len(tokenizer.word_index)} palabras")

## 5. Construcción de la red LSTM

Arquitectura:

```
Entrada (160 índices)
  -> Embedding(128)        # cada palabra se representa con 128 números
  -> SpatialDropout1D(0.2) # regularización palabra a palabra
  -> LSTM(128)             # memoria a corto y medio plazo
  -> Dense(64, relu)
  -> Dropout(0.5)
  -> Dense(1, sigmoid)     # probabilidad de spam
```

La capa `Embedding` es clave: una red recurrente no entiende palabras, solo
números. El embedding aprende una representación vectorial para cada palabra
durante el propio entrenamiento.

In [ ]:
modelo = keras.Sequential(
    [
        keras.Input(shape=(MAX_LEN,)),
        layers.Embedding(input_dim=min(len(tokenizer.word_index) + 1, MAX_VOCAB),
                         output_dim=EMBEDDING_DIM, name="embedding"),
        layers.SpatialDropout1D(0.2, name="spatial_dropout"),
        layers.LSTM(LSTM_UNITS, dropout=0.5, name="lstm"),
        layers.Dense(64, activation="relu", name="densa"),
        layers.Dropout(0.5, name="dropout"),
        layers.Dense(1, activation="sigmoid", name="salida"),
    ],
    name="clasificador_spam_lstm",
)

modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
)

modelo.summary()

## 6. Entrenamiento

Callbacks usados:

- `ModelCheckpoint` sobre `val_loss`: guarda la mejor época.
- `EarlyStopping(patience=4)`: para si la validación deja de mejorar.
- `ReduceLROnPlateau`: reduce la tasa de aprendizaje cuando la validación se estanca.

Las métricas se calculan después con scikit-learn, por lo que el modelo se
compila sin métricas de Keras.

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("spam_lstm.keras", monitor="val_loss",
                                   save_best_only=True),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4,
                                  restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                      patience=2, min_lr=1e-6, verbose=1),
]

historial = modelo.fit(
    x_train,
    y_train,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=2,
)

modelo = keras.models.load_model("spam_lstm.keras")
print("Mejor época:", int(np.argmin(historial.history["val_loss"])) + 1)

### Curvas de entrenamiento

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(historial.history["loss"], label="Entrenamiento", color="#4c72b0")
ax.plot(historial.history["val_loss"], label="Validación", color="#c44e52")
ax.set_title("Loss por época")
ax.set_xlabel("Época")
ax.set_ylabel("Binary crossentropy")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Evaluación

En un detector de spam los dos errores tienen costes distintos:

- **Falso positivo**: marcar un mensaje legítimo como spam (molesto, pero tolerable).
- **Falso negativo**: dejar pasar un spam (el coste que de verdad queremos evitar).

Por eso vigilamos especialmente el **recall** de la clase `spam`.

In [ ]:
probabilidades = modelo.predict(x_test, verbose=0).ravel()
predicciones = (probabilidades >= 0.5).astype(int)

accuracy = accuracy_score(y_test, predicciones)
precision, recall, f1, _ = precision_recall_fscore_support(
    y_test, predicciones, average="binary", zero_division=0
)
roc_auc = roc_auc_score(y_test, probabilidades)

print(classification_report(y_test, predicciones, target_names=CLASS_NAMES,
                             zero_division=0))
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision (spam): {precision:.4f}")
print(f"Recall (spam):    {recall:.4f}")
print(f"F1 (spam):        {f1:.4f}")
print(f"ROC-AUC:          {roc_auc:.4f}")

In [ ]:
cm = confusion_matrix(y_test, predicciones)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1)
ax1.set_title("Matriz de confusión")

cm_norm = cm / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", cbar=False,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax2)
ax2.set_title("Matriz normalizada por fila")
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_test, probabilidades)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, color="#c44e52", label=f"LSTM (AUC = {roc_auc:.4f})")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Azar")
ax.set_xlabel("Falsos positivos")
ax.set_ylabel("Verdaderos positivos")
ax.set_title("Curva ROC")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 8. Ejemplos de predicciones

In [ ]:
def predecir(texto):
    limpio = limpiar(texto)
    vector = codificar([limpio])
    return float(modelo.predict(vector, verbose=0)[0][0])


seleccion = np.random.default_rng(SEED).permutation(len(x_test))[:10]

filas = []
for indice in seleccion:
    texto = mensajes[idx_test[indice]]
    probabilidad = probabilidades[indice]
    predicho = "spam" if probabilidad >= 0.5 else "ham"
    filas.append(
        {
            "texto": texto[:70],
            "real": CLASS_NAMES[int(y_test[indice])],
            "predicho": predicho,
            "prob. spam": round(probabilidad, 4),
            "acierto": predicho == CLASS_NAMES[int(y_test[indice])],
        }
    )

import pandas as pd

tabla = pd.DataFrame(filas)
display(tabla)
print(f"Aciertos: {tabla['acierto'].sum()}/{len(tabla)}")

In [ ]:
mensajes_prueba = [
    "Hey how are you? Are we still meeting for lunch at 1pm?",
    "Ok lar... Joking wif u oni...",
    "Free entry in 2 a wkly comp to win FA Cup final tkts. Text FA to 87121",
    "URGENT! You have won a 2000 pound cash prize. Call 0871-872-9758 now",
]

for texto in mensajes_prueba:
    probabilidad = predecir(texto)
    etiqueta = "SPAM" if probabilidad >= 0.5 else "HAM"
    print(f"[{etiqueta}] {probabilidad:.2%}  <-  {texto[:65]}")

## 9. Conclusiones

- Una sola capa **LSTM** con *embedding* aprendido alcanza una accuracy cercana al
  99 % sobre el conjunto de prueba, muy por encima de la línea base (13.4 % de
  mensajes son spam, así que predecir siempre "ham" ya daría 86.6 %).
- El **recall** de la clase spam (~0.95) es la métrica clave: deja pasar menos
  de 6 de cada 100 mensajes de spam en el conjunto de prueba.
- El corpus está muy desbalanceado (13.4 % de spam), por lo que la accuracy
  por sí sola engaña: un modelo que dijera siempre "ham" saca 86.6 %.
- El entrenamiento converge rápido: el `EarlyStopping` detiene el proceso en
  pocas épocas porque el corpus es pequeño.
- Limitación: el modelo se ha entrenado sobre mensajes en inglés de principios
  de los 2000, escritos en clave y sin acentos. Funciona bien con spam actual,
  pero con español u otros idiomas habría que reentrenarlo.